<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/KREA2-LORA-TRAINING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Krea 2 LoRA training (A100 / L4)

Trains a LoRA for **Krea 2** with [ai-toolkit](https://github.com/ostris/ai-toolkit) — by default on **your Krea 2 Realism**
model (the one the Krea 2 notebook uses), with Ostris' Krea 2 Turbo training adapter so the LoRA works with fast 8-step
models. The finished LoRA is saved to `MyDrive/ComfyUI/loras_krea2`, so it shows up in the Krea 2 notebook's LoRA list.

| | |
|---|---|
| **What you give it** | 15–30 pictures of one person or character (+ a trigger word), or of a style |
| **GPU** | **A100** (1024 px, about 1–2 hours for a face LoRA) or **L4** (768 px, several times slower) |

**With the website**
1. **Runtime → Change runtime type → A100 GPU** (or L4). Don't run another GPU notebook at the same time.
2. Run **cell 1** (setup + downloads, about 10–15 minutes; Colab asks for Google Drive access) and **cell 2** (prints a `READY:` URL).
3. On the website open **LoRA Training**, paste the URL (the tab switches to Krea 2), name the LoRA, set a trigger word, add
   pictures, press **Auto-caption all** (or write captions), check them, and press **Start training**.
4. Keep this Colab tab open; the website shows the progress. Then start the **Krea 2** notebook, pick the LoRA in the
   LoRA list and use the trigger word in the prompt.

Progress is saved every 200 steps: if Colab disconnects, run cells 1–2 again and press Start training with the same name
to continue. **Cell 4** shows the progress here too. **Without the website:** cell 3 trains a LoRA from a folder.

Tips: Krea 2 identity LoRAs learn fast and can get stiff — 1,000–1,500 steps for 15–30 pictures is usually enough. Use
clear, varied pictures (angles, expressions, light, clothes). Only train on pictures you have the rights to use, and
only make a LoRA of a real person with that person's consent.

In [ ]:
# Cell 1: setup — Google Drive, ai-toolkit, the Krea 2 base model, and the shared training code
#@markdown **Base model** — what the LoRA is trained on:
#@markdown - `my_krea2_realism`: your Krea 2 Realism model (the one the Krea 2 notebook uses). Best match for your pictures; no Hugging Face account needed.
#@markdown - `official_krea2_turbo`: Krea's official Krea 2 Turbo. Needs a free Hugging Face account: accept the licence on huggingface.co/krea/Krea-2-Turbo, then add your token as a Colab secret named `HF_TOKEN` (🔑 on the left).
BASE_MODEL = "my_krea2_realism"  #@param ["my_krea2_realism", "official_krea2_turbo"]
%cd /content
import os, re, io, json, glob, time, shutil, signal, hashlib, struct, threading, subprocess, collections


def mount_google_drive():
    """Connects Google Drive (needed: pictures and LoRAs are kept there). Retries once after a clean unmount."""
    from google.colab import drive
    mp = "/content/drive"
    if os.path.isdir(mp) and not os.path.ismount(mp) and os.listdir(mp):
        keep = f"/content/drive_leftovers_{int(time.time())}"
        shutil.move(mp, keep)
        print(f"Moved files that were in Google Drive's way to {keep}")
    for attempt in (1, 2):
        try:
            drive.mount(mp, force_remount=True)  # Colab asks you to allow access
            return
        except Exception as e:
            print(f"Google Drive try {attempt} failed ({e})" + (" — trying once more..." if attempt == 1 else ""))
            try:
                drive.flush_and_unmount()
            except Exception:
                pass
            time.sleep(3)
    raise RuntimeError("Google Drive did not connect. Runtime → Disconnect and delete runtime, run this cell again, sign in "
                       "with the same Google account as Colab and tick every box on Google's page.")


mount_google_drive()
DRIVE_ROOT = "/content/drive/MyDrive/ComfyUI"
TRAIN_ROOT = f"{DRIVE_ROOT}/training_krea2"   # <name>/images (+ captions), <name>/output (saved progress)
LORAS_KREA2 = f"{DRIVE_ROOT}/loras_krea2"     # finished LoRAs go here — the Krea 2 notebook's LoRA folder
DRIVE_MODELS = f"{DRIVE_ROOT}/krea2_models"   # where the Krea 2 notebook keeps its model files
os.makedirs(TRAIN_ROOT, exist_ok=True)
os.makedirs(LORAS_KREA2, exist_ok=True)

# ---------------------------------------------------------------- GPU check
# Krea 2 is a 12B model: training needs a bf16 GPU with 20+ GB.
#   A100 (40/80 GB): 1024 px — about 1–2 hours for a face LoRA
#   L4 (22.5 GB):    768 px, model in 8-bit with low-VRAM mode — several times slower
try:
    _gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
                          capture_output=True, text=True).stdout.strip().split("\n")[0]
except FileNotFoundError:
    _gpu = ""
if _gpu.count(",") < 2:
    raise RuntimeError("This Colab runtime has no GPU, and training needs one.\n"
                       "Fix: Runtime → Change runtime type → A100 (or L4) GPU → Save, then run the cells again.")
_name, _mib, _cap = [x.strip() for x in _gpu.rsplit(",", 2)]
GPU = {"name": _name, "vram_gib": round(int(_mib) / 1024, 1), "compute_cap": _cap}
if float(_cap or 0) < 8.0 or GPU["vram_gib"] < 20:
    raise RuntimeError(f"This GPU ({GPU['name']}, {GPU['vram_gib']} GB) is too small for Krea 2 training.\n"
                       "Pick Runtime → Change runtime type → A100 GPU (best) or L4 GPU (slower), then run the cells again.")
if GPU["vram_gib"] >= 70:
    PROFILE = {"label": "big GPU", "resolution": 1024, "quantize": False, "low_vram": False}
elif GPU["vram_gib"] >= 36:
    PROFILE = {"label": "A100", "resolution": 1024, "quantize": True, "low_vram": False}
else:
    PROFILE = {"label": "L4", "resolution": 768, "quantize": True, "low_vram": True}
GPU["mode"] = "improved"
GPU["mode_label"] = (f"{PROFILE['label']}: Krea 2 LoRAs at {PROFILE['resolution']} px"
                     + (" — slow (model in 8-bit, low-VRAM mode); an A100 is several times faster" if PROFILE["low_vram"] else ""))
print(f"GPU: {GPU['name']} ({GPU['vram_gib']} GB) -> {GPU['mode_label']}")

# ---------------------------------------------------------------- install ai-toolkit (pinned)
AITK = "/content/ai-toolkit"
AITK_COMMIT = "7a5fe1a8ca72276e1376ce6577f853b480c45576"
if not os.path.exists(AITK):
    !git clone -q https://github.com/ostris/ai-toolkit {AITK}
!cd {AITK} && (git checkout -q {AITK_COMMIT} 2>/dev/null || (git fetch -q origin && git checkout -q {AITK_COMMIT})) && git submodule update --init --recursive -q
!cd {AITK} && pip install -q -r requirements.txt
!pip install -q fastapi uvicorn nest_asyncio
!apt install -qqy aria2 > /dev/null
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

# ---------------------------------------------------------------- the base model
# The text encoder (Qwen3-VL 4B) and the VAE are downloaded by ai-toolkit from Hugging Face on the first run.
TURBO_ADAPTER = "ostris/krea2_turbo_training_adapter/krea2_turbo_training_adapter_v1.safetensors"
EXPECTED_ROOTS = ("first.", "blocks.", "last.", "tmlp.", "tproj.", "txtfusion.", "txtmlp.")


def safetensors_keys(path):
    with open(path, "rb") as fh:
        n = struct.unpack("<Q", fh.read(8))[0]
        return [k for k in json.loads(fh.read(n)) if k != "__metadata__"]


def prepare_local_model(path):
    """ai-toolkit wants Krea's own key names (first.*, blocks.N.*, ...). ComfyUI files sometimes put a prefix in front
    (model.diffusion_model. / diffusion_model.): then a copy without it is written once."""
    keys = safetensors_keys(path)
    if any(k.startswith(EXPECTED_ROOTS) for k in keys):
        return path
    for prefix in ("model.diffusion_model.", "diffusion_model.", "model."):
        if sum(k.startswith(prefix) for k in keys) > len(keys) * 0.9 and any(k[len(prefix):].startswith(EXPECTED_ROOTS) for k in keys):
            out = "/content/krea2_base/" + os.path.basename(path).replace(".safetensors", "_aitk.safetensors")
            if not os.path.exists(out):
                print(f"Renaming the model's layers for ai-toolkit (once, a few minutes): {prefix}...")
                from safetensors import safe_open
                from safetensors.torch import save_file
                tensors, meta = {}, None
                with safe_open(path, "pt") as f:
                    meta = f.metadata()
                    for k in f.keys():
                        tensors[k[len(prefix):] if k.startswith(prefix) else k] = f.get_tensor(k)
                save_file(tensors, out + ".part", metadata=meta)
                os.replace(out + ".part", out)
                del tensors
            return out
    raise RuntimeError("This Krea 2 model file has a layout ai-toolkit doesn't know. Set BASE_MODEL to "
                       "official_krea2_turbo (needs a Hugging Face token) and run this cell again.")


if BASE_MODEL == "official_krea2_turbo":
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        raise RuntimeError("Add your Hugging Face token as a Colab secret named HF_TOKEN (🔑 on the left, allow this "
                           "notebook to use it), after accepting the licence on huggingface.co/krea/Krea-2-Turbo.")
    BASE = {"name_or_path": "krea/Krea-2-Turbo", "label": "official Krea 2 Turbo"}
else:
    os.makedirs("/content/krea2_base", exist_ok=True)
    local = "/content/krea2_base/Krea2Realism_V1.safetensors"
    cached = f"{DRIVE_MODELS}/Krea2Realism_V1.safetensors"
    if not os.path.exists(local):
        if os.path.exists(cached):
            print("Copying your Krea 2 Realism model from Google Drive...")
            shutil.copy(cached, local + ".part")
            os.replace(local + ".part", local)
        else:
            print("Downloading Krea 2 Realism (Civitai)...")
            subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "8", "-s", "8", "-k", "1M",
                            "https://civitai.red/api/download/models/3405129?fileId=3296411&token=777a5632707c15ecf54e13eb3b1b862b",
                            "-d", "/content/krea2_base", "-o", "Krea2Realism_V1.safetensors"], check=True)
    BASE = {"name_or_path": prepare_local_model(local), "label": "your Krea 2 Realism"}
print(f"Base model: {BASE['label']} (+ the Krea 2 Turbo training adapter, so the LoRA works with fast 8-step models)")

# ---------------------------------------------------------------- shared training code
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
DEFAULTS = {
    "trigger_word": "",
    "class_word": "man",
    "add_trigger_to_captions": True,
    "max_train_steps": 1200,
    "num_repeats": 1,
    "train_batch_size": 1,
    "resolution": PROFILE["resolution"],
    "network_dim": 16,
    "network_alpha": 16,
    "learning_rate": 1e-4,
    "seed": 42,
    "save_every_n_steps": 200,
}


def clean_name(name):
    return re.sub(r"[^\w\-]+", "_", (name or "").strip()).strip("_")


def clean_file(file):
    file = os.path.basename(file or "")
    stem, ext = os.path.splitext(file)
    stem = re.sub(r"[^\w\-. ]+", "_", stem).strip(" .") or "image"
    return stem + ext.lower()


def images_dir(name):
    n = clean_name(name)
    if not n:
        raise ValueError("Give the LoRA a name first")
    return f"{TRAIN_ROOT}/{n}/images"


def output_dir(name):
    return f"{TRAIN_ROOT}/{clean_name(name)}/output"


def list_images(name):
    d = images_dir(name)
    return sorted(f for f in os.listdir(d) if f.lower().endswith(IMAGE_EXTS)) if os.path.isdir(d) else []


def caption_path(name, file):
    return os.path.join(images_dir(name), os.path.splitext(clean_file(file))[0] + ".txt")


def read_caption(name, file):
    p = caption_path(name, file)
    return open(p, encoding="utf-8").read().strip() if os.path.exists(p) else ""


def write_caption(name, file, caption):
    with open(caption_path(name, file), "w", encoding="utf-8") as fh:
        fh.write((caption or "").strip())


def split_tags(caption):
    return [t.strip() for t in (caption or "").split(",") if t.strip()]


def list_datasets():
    out = []
    for n in sorted(os.listdir(TRAIN_ROOT)):
        if os.path.isdir(os.path.join(TRAIN_ROOT, n, "images")):
            out.append({"name": n, "images": len(list_images(n)), "kind": "character",
                        "has_lora": os.path.exists(f"{LORAS_KREA2}/{n}.safetensors")})
    return out


def saved_steps(name):
    """Steps of the saved checkpoints (ai-toolkit: <name>_000000200.safetensors)."""
    d = f"{output_dir(name)}/{clean_name(name)}"
    steps = []
    for p in glob.glob(f"{d}/{clean_name(name)}_*.safetensors"):
        m = re.search(r"_(\d{9})\.safetensors$", p)
        if m:
            steps.append(int(m.group(1)))
    return sorted(steps)


def saved_progress(name):
    steps = saved_steps(name)
    return steps[-1] if steps else None


def start_fresh(name):
    """Move the previous run aside so the next training starts from zero (nothing is deleted)."""
    n = clean_name(name)
    if busy() and STATE["name"] == n:
        raise RuntimeError(f"'{n}' is training right now — stop it first")
    base = f"{TRAIN_ROOT}/{n}"
    v = 1
    while os.path.exists(f"{base}/output_v{v}") or os.path.exists(f"{LORAS_KREA2}/{n}_v{v}.safetensors"):
        v += 1
    moved = []
    if os.path.isdir(f"{base}/output") and os.listdir(f"{base}/output"):
        os.rename(f"{base}/output", f"{base}/output_v{v}")
        moved.append(f"training_krea2/{n}/output → output_v{v}")
    if os.path.exists(f"{LORAS_KREA2}/{n}.safetensors"):
        os.rename(f"{LORAS_KREA2}/{n}.safetensors", f"{LORAS_KREA2}/{n}_v{v}.safetensors")
        moved.append(f"loras_krea2/{n}.safetensors → {n}_v{v}.safetensors")
    return {"moved": moved, "version": v if moved else None}


# ---- one background task at a time (training or auto-captioning), shared with the website
STATE = {"status": "idle", "task": None, "name": None, "phase": "", "step": 0, "total": 0,
         "elapsed": "", "eta": "", "error": None, "result": None, "started": None,
         "log": collections.deque(maxlen=60)}
_PROC = {"p": None, "stop": False, "steps": 0}
# ai-toolkit's progress bar: "<name>:  12%|██  | 240/2000 [10:00<1:10:00, 2.50s/it, lr: 1.0e-04 loss: 3.1e-01]"
PROGRESS_RE = re.compile(r"\|\s*(\d+)/(\d+)\s*\[([^<\]]*)<([^,\]]*)")


def status_payload():
    s = {k: v for k, v in STATE.items() if k != "log"}
    s["log"] = list(STATE["log"])[-25:]
    return s


def _log(line):
    line = line.rstrip()
    if line:
        STATE["log"].append(line)


def _run(cmd, cwd, env_extra=None):
    """Run a command, stream its output into STATE (the training progress bar updates step/eta)."""
    env = {**os.environ, "PYTHONUNBUFFERED": "1", **(env_extra or {})}
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0, env=env,
                         start_new_session=True)  # stopping a notebook cell must not stop training
    _PROC["p"] = p
    buf = b""
    while True:
        chunk = p.stdout.read(4096)
        if not chunk:
            break
        buf += chunk
        parts = re.split(rb"[\r\n]", buf)
        buf = parts.pop()
        for raw in parts:
            line = raw.decode("utf-8", "replace").strip()
            if not line:
                continue
            m = PROGRESS_RE.search(line)
            if m and _PROC["steps"] and int(m.group(2)) == _PROC["steps"]:   # the training bar (not the caching bars)
                STATE.update(step=int(m.group(1)), total=int(m.group(2)),
                             elapsed=m.group(3).strip(), eta=m.group(4).strip(), phase="Training")
            elif not m:
                _log(line)
    rc = p.wait()
    _PROC["p"] = None
    return rc


def busy():
    return STATE["status"] == "running"


def _begin(task, name, phase):
    if busy():
        raise RuntimeError(f"Busy: {STATE['task']} of '{STATE['name']}' is still running")
    STATE["log"].clear()
    STATE.update(status="running", task=task, name=clean_name(name), phase=phase, step=0, total=0,
                 elapsed="", eta="", error=None, result=None, started=time.time())
    _PROC["stop"] = False


def _finish(error=None, result=None):
    if _PROC["stop"]:
        STATE.update(status="stopped", phase="Stopped")
    elif error:
        STATE.update(status="error", error=str(error), phase="Failed")
    else:
        STATE.update(status="done", result=result, phase="Done")


def stop_task():
    _PROC["stop"] = True
    p = _PROC["p"]
    if p and p.poll() is None:
        try:
            os.killpg(p.pid, signal.SIGTERM)
        except Exception:
            p.terminate()


def prepare_dataset(name, s):
    """Copy the pictures + captions to the runtime. Captions start with '<trigger> <class>' so the trigger word learns
    the person (a trigger word already at the start, e.g. from Auto-caption, isn't repeated)."""
    imgs = list_images(name)
    if len(imgs) < 4:
        raise RuntimeError(f"Add more pictures of '{name}' first — at least 4 (15-30 works best)")
    local = f"/content/dataset/{clean_name(name)}"
    shutil.rmtree(local, ignore_errors=True)
    os.makedirs(local)
    trigger, cls = s["trigger_word"].strip(), s["class_word"].strip()
    who = " ".join(w for w in (trigger, cls) if w)
    for f in imgs:
        stem = os.path.splitext(f)[0]
        shutil.copy2(os.path.join(images_dir(name), f), f"{local}/{f}")
        caption = read_caption(name, f)
        rest = caption
        if trigger and rest.lower().startswith(trigger.lower()):
            rest = rest[len(trigger):].lstrip(" ,")
        if (s["add_trigger_to_captions"] or not caption) and who:
            caption = who + (f", {rest}" if rest else "")
        with open(f"{local}/{stem}.txt", "w", encoding="utf-8") as fh:
            fh.write(caption or who or "a photo")
    return local, len(imgs)


def build_config(name, s, dataset):
    """ai-toolkit config. It resumes by itself from the newest checkpoint in the output folder."""
    n, out = clean_name(name), output_dir(name)
    os.makedirs(out, exist_ok=True)
    saved_path = f"{out}/run_settings.json"
    must_match = {k: s[k] for k in ("network_dim", "network_alpha", "resolution", "max_train_steps")}
    must_match["base"] = BASE["label"]
    if saved_steps(name) and os.path.exists(saved_path):
        saved = json.load(open(saved_path))
        diff = [f"{k}: {saved.get(k)} → {v}" for k, v in must_match.items() if saved.get(k) != v]
        if diff:
            raise RuntimeError(f"'{n}' already has saved progress with different settings ({'; '.join(diff)}). "
                               f"Press 'Start fresh' on the website (or use a new LoRA name) to train with the new "
                               f"settings, or switch back to the old settings to continue.")
    json.dump(must_match, open(saved_path, "w"))
    res = s["resolution"]
    model = {"name_or_path": BASE["name_or_path"], "arch": "krea2", "quantize": PROFILE["quantize"], "quantize_te": True,
             "low_vram": PROFILE["low_vram"], "assistant_lora_path": TURBO_ADAPTER}
    cfg = {
        "job": "extension",
        "config": {
            "name": n,
            "process": [{
                "type": "sd_trainer",
                "training_folder": out,
                "device": "cuda:0",
                "network": {"type": "lora", "linear": s["network_dim"], "linear_alpha": s["network_alpha"]},
                "save": {"dtype": "bf16", "save_every": s["save_every_n_steps"], "max_step_saves_to_keep": 4},
                "datasets": [{
                    "folder_path": dataset, "caption_ext": "txt", "caption_dropout_rate": 0.05,
                    "shuffle_tokens": False, "cache_latents_to_disk": True, "num_repeats": s["num_repeats"],
                    "resolution": [r for r in (512, 768, 1024) if r <= res] or [res],
                }],
                "train": {
                    "batch_size": 1, "steps": s["max_train_steps"], "gradient_accumulation": 1,
                    "train_unet": True, "train_text_encoder": False, "gradient_checkpointing": True,
                    "noise_scheduler": "flowmatch", "timestep_type": "linear", "optimizer": "adamw8bit",
                    "lr": s["learning_rate"], "dtype": "bf16", "cache_text_embeddings": True,
                    "disable_sampling": True, "seed": s["seed"],
                },
                "model": model,
            }],
        },
        "meta": {"name": "[name]", "version": "1.0"},
    }
    path = f"/content/dataset/{n}_config.json"
    json.dump(cfg, open(path, "w"), indent=1)
    return path


def merge_settings(settings):
    s = dict(DEFAULTS)
    for k, v in (settings or {}).items():
        if k in DEFAULTS and v is not None and v != "":
            t = type(DEFAULTS[k])
            if t is bool:
                s[k] = v if isinstance(v, bool) else str(v).lower() in ("1", "true", "yes")
            elif t is int:
                s[k] = int(float(v))
            else:
                s[k] = t(v)
    s["train_batch_size"] = 1
    s["resolution"] = min(max(512, s["resolution"] // 64 * 64), 1024)
    s["max_train_steps"] = max(50, s["max_train_steps"])
    s["save_every_n_steps"] = max(50, s["save_every_n_steps"])
    s["num_repeats"] = max(1, s["num_repeats"])
    return s


def _train_worker(name, s):
    try:
        if s["resolution"] > PROFILE["resolution"]:
            _log(f"{GPU['name']}: training at {PROFILE['resolution']} px ({s['resolution']} doesn't fit)")
            s["resolution"] = PROFILE["resolution"]
        STATE["phase"] = "Preparing pictures"
        dataset, n = prepare_dataset(name, s)
        _log(f"Krea 2 LoRA: {n} pictures, {s['max_train_steps']} steps, {s['resolution']} px, GPU {GPU['name']}, base: {BASE['label']}")
        config = build_config(name, s, dataset)
        if saved_progress(name):
            _log(f"Continuing from step {saved_progress(name)}")
        STATE.update(phase="Loading the models and caching the pictures (the first run downloads ~10 GB: 10–15 min)",
                     total=s["max_train_steps"])
        _PROC["steps"] = s["max_train_steps"]
        rc = _run(["python", "run.py", config], AITK, {"HF_HUB_ENABLE_HF_TRANSFER": "1"})
        final = f"{output_dir(name)}/{clean_name(name)}/{clean_name(name)}.safetensors"
        if rc == 0 and os.path.exists(final):
            dest = f"{LORAS_KREA2}/{clean_name(name)}.safetensors"
            shutil.copy2(final, dest)
            _finish(result={"lora": os.path.basename(dest), "path": dest})
        else:
            tail = " | ".join(list(STATE["log"])[-3:])
            hint = " (out of GPU memory: try Resolution 768, or an A100)" if "out of memory" in tail.lower() else ""
            _finish(error=f"Training failed (exit code {rc}){hint}. Last output: {tail}")
    except Exception as e:
        _finish(error=e)


def start_training(name, settings=None):
    s = merge_settings(settings)
    _begin("training", name, "Starting")
    STATE["settings"] = s
    threading.Thread(target=_train_worker, args=(name, s), daemon=True).start()
    return s


# ---- Auto-caption: one or two sentences per picture with Qwen3-VL 4B (Krea 2's own text encoder), run as its own
# program so its memory is freed afterwards.
CAPTION_PROMPT = ("Describe this image in one or two sentences for a training dataset: what the person is doing, their "
                  "pose, clothing, the setting, framing and lighting. Do not describe the face, hair colour or eye "
                  "colour, and do not guess who it is.")
CAPTION_SCRIPT = "/content/krea2_caption.py"
open(CAPTION_SCRIPT, "w").write('''
import sys, os, json, torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor
folder, prompt = sys.argv[1], sys.argv[2]
name = "Qwen/Qwen3-VL-4B-Instruct"
model = AutoModelForImageTextToText.from_pretrained(name, torch_dtype=torch.bfloat16, device_map="cuda")
proc = AutoProcessor.from_pretrained(name)
files = sorted(f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp", ".bmp")))
for i, f in enumerate(files):
    img = Image.open(os.path.join(folder, f)).convert("RGB")
    img.thumbnail((896, 896))
    msgs = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]}]
    inputs = proc.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
    text = proc.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()
    open(os.path.join(folder, os.path.splitext(f)[0] + ".txt"), "w", encoding="utf-8").write(" ".join(text.split()))
    print(f"captioned {i + 1}/{len(files)}: {f}", flush=True)
''')


def _autotag_worker(name, mode, trigger):
    try:
        imgs = list_images(name)
        if not imgs:
            raise RuntimeError("No pictures to caption")
        work = f"/content/autocaption/{clean_name(name)}"
        shutil.rmtree(work, ignore_errors=True)
        os.makedirs(work)
        for f in imgs:
            shutil.copy2(os.path.join(images_dir(name), f), os.path.join(work, f))
        STATE["phase"] = "Auto-captioning with Qwen3-VL (the first run downloads it: a few minutes)"
        if _run(["python", CAPTION_SCRIPT, work, CAPTION_PROMPT], "/content") != 0:
            raise RuntimeError("The captioner failed — see the log")
        changed = 0
        for f in imgs:
            p = os.path.join(work, os.path.splitext(f)[0] + ".txt")
            if not os.path.exists(p):
                continue
            new = " ".join(open(p, encoding="utf-8").read().split())
            old = read_caption(name, f)
            caption = f"{old}, {new}" if mode == "append" and old else new
            if trigger and trigger.lower() not in caption.lower():
                caption = f"{trigger}, {caption}"
            write_caption(name, f, caption)
            changed += 1
        _finish(result={"tagged": changed})
    except Exception as e:
        _finish(error=e)


def start_autotag(name, threshold=0.35, mode="replace", trigger="", exclude=""):
    """Sentence captions with Qwen3-VL (threshold / exclude are for the SDXL tagger; not used here)."""
    _begin("auto-tagging", name, "Starting the captioner")
    threading.Thread(target=_autotag_worker, args=(name, mode, (trigger or "").strip()), daemon=True).start()


print("Setup done. Run cell 2 to use the website, or cell 3 to train without it.")


In [ ]:
# Cell 2: training server for the website (LoRA Training tab → it switches to Krea 2 by itself). Paste the READY: URL there.
# This cell finishes after printing the URL; the server keeps running in the background.
if "clean_name" not in globals():
    raise RuntimeError("Run cell 1 (setup) first, then run this cell again.")

import base64, io, re, subprocess, threading, time
import nest_asyncio, uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from pydantic import BaseModel
from PIL import Image, ImageOps

nest_asyncio.apply()
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True,
                   allow_methods=["*"], allow_headers=["*"])


def fail(msg, code=400):
    raise HTTPException(status_code=code, detail=str(msg))


def dataset_payload(name):
    return {"name": clean_name(name), "kind": "character",
            "images": [{"file": f, "caption": read_caption(name, f), "control": None} for f in list_images(name)],
            "saved_step": saved_progress(name),
            "has_lora": os.path.exists(f"{LORAS_KREA2}/{clean_name(name)}.safetensors")}


@app.get("/")
def root():
    return {"status": "ok", "mode": "lora_training"}


@app.get("/training/info")
def info():
    return {"mode": "lora_training", "trainer": "krea2", "defaults": DEFAULTS, "datasets": list_datasets(),
            "status": status_payload(), "gpu": {**GPU, "base": BASE["label"]}, "kinds": ["character"]}


@app.get("/training/status")
def status():
    return status_payload()


@app.get("/training/dataset/{name}")
def get_dataset(name: str):
    try:
        return dataset_payload(name)
    except ValueError as e:
        fail(e)


class Upload(BaseModel):
    filename: str
    data: str  # base64 (data: URL prefix allowed)
    caption: str | None = None


@app.post("/training/dataset/{name}/upload")
def upload(name: str, req: Upload):
    try:
        d = images_dir(name)
    except ValueError as e:
        fail(e)
    file = clean_file(req.filename)
    if not file.lower().endswith(IMAGE_EXTS):
        fail(f"{req.filename}: only jpg, png, webp and bmp images are supported")
    raw = base64.b64decode(re.sub(r"^data:[^,]*,", "", req.data))
    try:
        Image.open(io.BytesIO(raw)).verify()
    except Exception:
        fail(f"{req.filename} is not a readable image")
    os.makedirs(d, exist_ok=True)
    stem, ext = os.path.splitext(file)
    i = 1
    while os.path.exists(os.path.join(d, file)):  # never overwrite: photo.jpg -> photo-1.jpg
        file, i = f"{stem}-{i}{ext}", i + 1
    with open(os.path.join(d, file), "wb") as fh:
        fh.write(raw)
    if req.caption is not None:
        write_caption(name, file, req.caption)
    return {"file": file, "caption": read_caption(name, file)}


class Caption(BaseModel):
    file: str
    caption: str


@app.put("/training/dataset/{name}/caption")
def set_caption(name: str, req: Caption):
    if clean_file(req.file) not in list_images(name):
        fail("Image not found", 404)
    write_caption(name, req.file, req.caption)
    return {"file": req.file, "caption": read_caption(name, req.file)}


class Bulk(BaseModel):
    action: str  # "add" (to the start) or "remove"
    tag: str


@app.post("/training/dataset/{name}/bulk")
def bulk(name: str, req: Bulk):
    tag = req.tag.strip().strip(",").strip()
    if not tag:
        fail("Type a tag first")
    for f in list_images(name):
        tags = [t for t in split_tags(read_caption(name, f)) if t.lower() != tag.lower()]
        if req.action == "add":
            tags.insert(0, tag)
        write_caption(name, f, ", ".join(tags))
    return dataset_payload(name)


@app.delete("/training/dataset/{name}/image/{file}")
def delete_image(name: str, file: str):
    file = clean_file(file)
    if file not in list_images(name):
        fail("Image not found", 404)
    os.remove(os.path.join(images_dir(name), file))
    if os.path.exists(caption_path(name, file)):
        os.remove(caption_path(name, file))
    return dataset_payload(name)


@app.get("/training/dataset/{name}/file/{file}")
def full_file(name: str, file: str):
    """The original image (for the website's cloud backup of the dataset)."""
    file = clean_file(file)
    src = os.path.join(images_dir(name), file)
    if file not in list_images(name) or not os.path.exists(src):
        fail("Image not found", 404)
    ext = os.path.splitext(file)[1].lower().lstrip(".")
    media = {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png", "webp": "image/webp", "bmp": "image/bmp"}.get(ext, "application/octet-stream")
    return Response(open(src, "rb").read(), media_type=media)


THUMBS = "/content/thumbs"


@app.get("/training/dataset/{name}/thumb/{file}")
def thumb(name: str, file: str):
    file = clean_file(file)
    src = os.path.join(images_dir(name), file)
    if not os.path.exists(src):
        fail("Image not found", 404)
    key = hashlib.sha1(f"{src}|{os.path.getmtime(src)}".encode()).hexdigest()
    path = os.path.join(THUMBS, key + ".jpg")
    if not os.path.exists(path):
        os.makedirs(THUMBS, exist_ok=True)
        im = ImageOps.exif_transpose(Image.open(src)).convert("RGB")
        im.thumbnail((360, 360))
        im.save(path, "JPEG", quality=85)
    return Response(open(path, "rb").read(), media_type="image/jpeg", headers={"Cache-Control": "max-age=3600"})


def _thumb_response(src):
    key = hashlib.sha1(f"{src}|{os.path.getmtime(src)}".encode()).hexdigest()
    path = os.path.join(THUMBS, key + ".jpg")
    if not os.path.exists(path):
        os.makedirs(THUMBS, exist_ok=True)
        im = ImageOps.exif_transpose(Image.open(src)).convert("RGB")
        im.thumbnail((360, 360))
        im.save(path, "JPEG", quality=85)
    return Response(open(path, "rb").read(), media_type="image/jpeg", headers={"Cache-Control": "max-age=3600"})


class AutoTag(BaseModel):
    threshold: float = 0.35
    mode: str = "replace"  # "append" to existing captions or "replace" them
    trigger_word: str = ""
    exclude_tags: str = ""  # comma-separated tags to leave out


@app.post("/training/dataset/{name}/autotag")
def autotag(name: str, req: AutoTag):
    try:
        start_autotag(name, req.threshold, req.mode, req.trigger_word, req.exclude_tags)
    except Exception as e:
        fail(e)
    return status_payload()


class Start(BaseModel):
    name: str
    settings: dict = {}


@app.post("/training/start")
def start(req: Start):
    try:
        start_training(req.name, req.settings)
    except Exception as e:
        fail(e)
    return status_payload()


@app.post("/training/dataset/{name}/start_fresh")
def fresh(name: str):
    try:
        result = start_fresh(name)
    except Exception as e:
        fail(e)
    return {**result, **dataset_payload(name)}


@app.post("/training/stop")
def stop():
    stop_task()
    return status_payload()


_tunnel = globals().get("TRAINING_TUNNEL")
if globals().get("TRAINING_SERVER_URL") and _tunnel is not None and _tunnel.poll() is None:
    print(f"The server is already running: {TRAINING_SERVER_URL}")
else:
    if globals().get("TRAINING_SERVER_URL"):
        print("The old tunnel stopped, so the old URL no longer works — starting a new one (new URL below).")
    if not globals().get("TRAINING_UVICORN_STARTED"):
        threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
                         daemon=True).start()
        TRAINING_UVICORN_STARTED = True
        time.sleep(4)
    cf = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                          start_new_session=True)  # survives stopping a notebook cell
    TRAINING_TUNNEL = cf
    url, t0 = None, time.time()
    while time.time() - t0 < 60:
        line = cf.stdout.readline()
        if not line:
            break
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0)
            break
    threading.Thread(target=lambda: [None for _ in cf.stdout], daemon=True).start()  # keep the pipe drained
    if url:
        TRAINING_SERVER_URL = url
        print("=" * 60)
        print(f"READY: {url}")
        print("Paste this into the website: LoRA Training tab → Training server URL (it switches to Krea 2 training).")
        print("Keep this Colab tab open while you work. Progress is also printed by cell 4 if you want it here.")
        print("=" * 60)
    else:
        print("Could not start the Cloudflare tunnel — run this cell again.")

In [ ]:
# Cell 3 (optional): train WITHOUT the website — fill in the form and run. Skip this if you use the website.
#@markdown Pictures: `MyDrive/ComfyUI/training_krea2/<name>/images` (a file picker opens if the folder is empty).
LORA_NAME = "my_character"  #@param {type:"string"}
TRIGGER_WORD = "zxq_me"  #@param {type:"string"}
CLASS_WORD = "man"  #@param {type:"string"}
MAX_TRAIN_STEPS = 1200  #@param {type:"integer"}
NETWORK_DIM = 16  #@param {type:"integer"}
LEARNING_RATE = 1e-4  #@param {type:"number"}

if "clean_name" not in globals():
    raise RuntimeError("Run cell 1 (setup) first, then run this cell again.")

import os, io, zipfile, time
from google.colab import files

name = clean_name(LORA_NAME)
d = images_dir(name)
os.makedirs(d, exist_ok=True)
if not list_images(name):
    print(f"Choose images, caption .txt files or one .zip. They'll be saved to {d}")
    for fname, data in files.upload().items():
        if fname.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                for m in z.namelist():
                    base = os.path.basename(m)
                    if base and not base.startswith(".") and "__MACOSX" not in m and base.lower().endswith(IMAGE_EXTS + (".txt",)):
                        open(os.path.join(d, clean_file(base)), "wb").write(z.read(m))
        elif fname.lower().endswith(IMAGE_EXTS + (".txt",)):
            open(os.path.join(d, clean_file(fname)), "wb").write(data)
        if os.path.exists(fname):
            os.remove(fname)
print(f"{len(list_images(name))} images in {d}")

start_training(name, {"trigger_word": TRIGGER_WORD, "class_word": CLASS_WORD, "max_train_steps": MAX_TRAIN_STEPS,
                      "network_dim": NETWORK_DIM, "network_alpha": NETWORK_DIM, "learning_rate": LEARNING_RATE})
last = None
while STATE["status"] == "running":
    msg = f"{STATE['phase']}" + (f" — step {STATE['step']}/{STATE['total']}, {STATE['eta']} left" if STATE["step"] else "")
    if msg != last:
        print(time.strftime("%H:%M"), msg)
        last = msg
    time.sleep(20)
print(STATE["status"], STATE.get("result") or STATE.get("error") or "")
if STATE["status"] == "done":
    print(f"Start the Krea 2 notebook and pick {STATE['result']['lora']} in the website's LoRA list (Krea 2 LoRAs); "
          f"use '{TRIGGER_WORD}' in the prompt.")

In [ ]:
# Cell 4 (optional): show the current task's progress and log here in Colab
if "clean_name" not in globals():
    raise RuntimeError("Run cell 1 (setup) first, then run this cell again.")

import time
from IPython.display import clear_output
try:
    while True:
        s = status_payload()
        clear_output(wait=True)
        print(f"{s['task'] or 'nothing running'} {s['name'] or ''}: {s['status']} — {s['phase']}")
        if s["total"]:
            print(f"step {s['step']}/{s['total']}  elapsed {s['elapsed']}  left {s['eta']}")
        print("\n".join(s["log"][-15:]))
        if s["status"] != "running":
            break
        time.sleep(10)
except KeyboardInterrupt:
    pass